In [1]:
import pymysql

#create a connection
conn = pymysql.connect(
    host = 'localhost',
    user = 'root',
    password = "sabi3601",
    database = 'nhl_analytics'
)

#create a cursor
cursor = conn.cursor()

In [3]:
cursor.execute("SELECT team_id, team_abbrev FROM teams")

team_abbrev_to_id = {abbrev: team_id for team_id, abbrev in cursor.fetchall()}

team_abbrevs = list(team_abbrev_to_id.keys())

len(team_abbrevs)

32

In [4]:
import requests

url = f"https://api-web.nhle.com/v1/roster/{team_abbrevs[0]}/current"

response = requests.get(url)

response.status_code # 200 indicates success

200

In [5]:
data = response.json()
data.keys() 
# forwards, defensemen, goalies

dict_keys(['forwards', 'defensemen', 'goalies'])

In [6]:
data['forwards'][0]

{'id': 8484153,
 'headshot': 'https://assets.nhle.com/mugs/nhl/20262027/ANA/8484153.png',
 'firstName': {'default': 'Leo'},
 'lastName': {'default': 'Carlsson'},
 'sweaterNumber': 91,
 'positionCode': 'C',
 'shootsCatches': 'L',
 'heightInInches': 75,
 'weightInPounds': 215,
 'heightInCentimeters': 191,
 'weightInKilograms': 98,
 'birthDate': '2004-12-26',
 'birthCity': {'default': 'Karlstad'},
 'birthCountry': 'SWE'}

In [11]:
first_name = data['forwards'][0]['firstName']['default']
last_name = data['forwards'][0]['lastName']['default']
position = data['forwards'][0]['positionCode']
jersey_number = data['forwards'][0]['sweaterNumber']
birth_date = data['forwards'][0]['birthDate']
birth_country = data['forwards'][0]['birthCountry']
height_cm = data['forwards'][0]['heightInCentimeters']
weight_kg = data['forwards'][0]['weightInKilograms']
shoots_catches = data['forwards'][0]['shootsCatches']
headshot_url = data['forwards'][0]['headshot']

In [7]:
import time

players = []

for abbrev in team_abbrevs:

    url = f"https://api-web.nhle.com/v1/roster/{abbrev}/current"
    response = requests.get(url)
    roster = response.json()

    for group in ['forwards', 'defensemen', 'goalies']:
        for p in roster[group]:
            players.append({
                'player_id': p['id'],
                'team_abbrev': abbrev,
                'first_name': p['firstName']['default'],
                'last_name': p['lastName']['default'],
                'position': p['positionCode'],
                'jersey_number': p.get('sweaterNumber'),
                'birth_date': p.get('birthDate'),
                'birth_country': p.get('birthCountry'),
                'height_cm': p.get('heightInCentimeters'),
                'weight_kg': p.get('weightInKilograms'),
                'shoots_catches': p.get('shootsCatches'),
                'headshot_url': p.get('headshot')
            })
    time.sleep(0.5)

print(f"{len(players)} players collected")

769 players collected


In [8]:
for row in players:
    row['team_id'] = team_abbrev_to_id[row['team_abbrev']]

players[0]

{'player_id': 8484153,
 'team_abbrev': 'ANA',
 'first_name': 'Leo',
 'last_name': 'Carlsson',
 'position': 'C',
 'jersey_number': 91,
 'birth_date': '2004-12-26',
 'birth_country': 'SWE',
 'height_cm': 191,
 'weight_kg': 98,
 'shoots_catches': 'L',
 'headshot_url': 'https://assets.nhle.com/mugs/nhl/20262027/ANA/8484153.png',
 'team_id': 18}

In [9]:
create_players_table = """
CREATE TABLE IF NOT EXISTS players (
    player_id BIGINT PRIMARY KEY,
    team_id INT,
    first_name VARCHAR(100) NOT NULL,
    last_name VARCHAR(100) NOT NULL,
    position VARCHAR(10),
    jersey_number INT,
    birth_date DATE,
    birth_country VARCHAR(10),
    height_cm FLOAT,
    weight_kg FLOAT,
    shoots_catches VARCHAR(5),
    headshot_url TEXT,
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
)
"""
cursor.execute(create_players_table)
conn.commit()
print("Players table created successfully")

Players table created successfully


In [11]:
insert_query = """INSERT IGNORE INTO players
                    (player_id,team_id,first_name,last_name,position,jersey_number,birth_date,birth_country,height_cm,weight_kg,shoots_catches,headshot_url)
                    VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)"""

for i in players:
   values = (i['player_id'],
         i['team_id'],
         i['first_name'],
         i['last_name'],
         i['position'],
         i['jersey_number'],
         i['birth_date'],
         i['birth_country'],
         i['height_cm'],
         i['weight_kg'],
         i['shoots_catches'],
         i['headshot_url'])
   cursor.execute(insert_query, values)

conn.commit()

print(f"{len(players)} players inserted successfully")

769 players inserted successfully


In [12]:
cursor.execute("SELECT COUNT(*) FROM players")
cursor.fetchone()

(1301,)